# SEBAL Notebook 3: Meteorological Data Acquisition from Analysis of Record for Calibration (AORC)
**Scene:** Landsat 7 (2020-01-19)  
**Region:** Mississippi Delta (EPSG:32615)

This notebook downloads meteorological forcing data from the **Analysis of Record for Calibration (AORC)** dataset for SEBAL processing over the Mississippi Delta. The workflow extracts hourly gridded variables at the Landsat overpass time and clips them to the study-area boundary. Retrieved variables include air temperature, specific humidity, wind components, incoming shortwave radiation, and incoming longwave radiation, which support aerodynamic resistance estimation and sensible heat flux calculations within the SEBAL energy balance framework.

## 1. Import Required Python Libraries and Utility Functions

In [1]:
import os
import subprocess, shlex, os, sys
import os, glob
from Utility import clip_save_met_var
import rasterio
import numpy as np

## 2. Input and directotories for this work

In [2]:
# date + region (match SEBAL workflow)
date = "20200119"
region = "MSDelta"
hour_str = "16"         # UTC hour (00–23)

# Directories
raw_met_dir = r"..\02_raw_aorc"
landsat_dir = r"..\03_clip_landsat"
# output folder
proc_dir = r"..\03_processed_met"


## 3. Run AORC downloader (using your script)

In [3]:
cmd = [
    sys.executable, "extract_aorc_hours_bbox.py",
    "--year", "2020",
    "--start", "2020-01-19 00:00:00",
    "--end",   "2020-01-19 23:00:00",
    "--bbox", "-92.5", "-88.0", "32.0", "36.0",
    "--outdir", raw_met_dir
]

print("Running:\n", " ".join(shlex.quote(x) for x in cmd))
p = subprocess.run(cmd, capture_output=True, text=True)

print("\nRETURN CODE:", p.returncode)
print("\nSTDOUT:\n", p.stdout[:4000])
print("\nSTDERR:\n", p.stderr[:4000])

Running:
 'C:\Users\mahes\.conda\envs\sebal_dl\python.exe' extract_aorc_hours_bbox.py --year 2020 --start '2020-01-19 00:00:00' --end '2020-01-19 23:00:00' --bbox -92.5 -88.0 32.0 36.0 --outdir '..\02_raw_aorc'

RETURN CODE: 0

STDOUT:
 [INFO] Zarr: s3://noaa-nws-aorc-v1-1-1km/2020.zarr
[INFO] BBOX(WGS84+buf): lon[-92.5500,-87.9500] lat[31.9500,36.0500]
[INFO] Hours: 24  (2020-01-19 00:00:00 -> 2020-01-19 23:00:00)
[INFO] OUT: ..\02_raw_aorc
[DONE]
[SUMMARY] wrote=0 skipped_exists=24 time_missing=0


STDERR:
 


## 4. Extract meteorological variablef for SEBAL

In [4]:
TA_tif = clip_save_met_var(raw_met_dir, landsat_dir, proc_dir, date, hour=hour_str, region='MSDelta', aorc_var='TMP_2maboveground', out_name='TA')
U_tif = clip_save_met_var(raw_met_dir, landsat_dir, proc_dir, date, hour=hour_str, region='MSDelta', aorc_var='UGRD_10maboveground', out_name='U10')
V_tif = clip_save_met_var(raw_met_dir, landsat_dir, proc_dir, date, hour=hour_str, region='MSDelta', aorc_var='VGRD_10maboveground', out_name='V10')
Rs_tif = clip_save_met_var(raw_met_dir, landsat_dir, proc_dir, date, hour=hour_str, region='MSDelta', aorc_var='DSWRF_surface', out_name='Rs')


Saved: ..\03_processed_met\TA_20200119_16Z_MSDelta.tif shape: (9905, 3618) var: TMP_2maboveground
Saved: ..\03_processed_met\U10_20200119_16Z_MSDelta.tif shape: (9905, 3618) var: UGRD_10maboveground
Saved: ..\03_processed_met\V10_20200119_16Z_MSDelta.tif shape: (9905, 3618) var: VGRD_10maboveground
Saved: ..\03_processed_met\Rs_20200119_16Z_MSDelta.tif shape: (9905, 3618) var: DSWRF_surface


## 5 Compute and save wind speed from U10 and V10

In [5]:
# Input rasters already created in this notebook
u10_path = U_tif
v10_path = V_tif

# Read U10 and V10
with rasterio.open(u10_path) as src_u, rasterio.open(v10_path) as src_v:
    u10 = src_u.read(1).astype("float32")
    v10 = src_v.read(1).astype("float32")
    profile = src_u.profile.copy()
    nodata_u = src_u.nodata
    nodata_v = src_v.nodata

# Build valid mask
valid = np.isfinite(u10) & np.isfinite(v10)
if nodata_u is not None:
    valid &= (u10 != nodata_u)
if nodata_v is not None:
    valid &= (v10 != nodata_v)

# Initialize output
wind = np.full(u10.shape, -9999.0, dtype="float32")

# Compute wind speed
wind[valid] = np.sqrt(u10[valid]**2 + v10[valid]**2)

# Save output
profile.update(dtype="float32", nodata=-9999.0)
wind_path = f"{proc_dir}/Wind10_{date}_{hour_str}Z_MSDelta.tif"

with rasterio.open(wind_path, "w", **profile) as dst:
    dst.write(wind, 1)

print("Saved:", wind_path)
print("Wind min:", np.nanmin(np.where(wind != -9999.0, wind, np.nan)))
print("Wind max:", np.nanmax(np.where(wind != -9999.0, wind, np.nan)))

Saved: ..\03_processed_met/Wind10_20200119_16Z_MSDelta.tif
Wind min: 0.44721362
Wind max: 8.18352
